# Experiment 4 — Term Frequency and Named Entity Recognition

This notebook implements all three sub-parts from the experiment sheet:

- **4.1:** Term Frequency + Named Entity Recognition using an NLP toolkit
- **4.2:** Term Frequency without using an NLP toolkit
- **4.3:** TF, DF, IDF and TF-IDF for multiple documents without NLTK, spaCy, or another NLP toolkit

> **Colab setup:** Upload your dataset(s) and change the path variables in the relevant section below.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4.1 — Term-Frequency Analysis and Named Entity Recognition (using a toolkit)

**Tasks covered:** calculate word frequencies, save `(Term, Frequency)` to CSV, display the top 10 terms, and perform NER.


In [1]:
!pip -q install spacy
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 69.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
INPUT_FILE_41 = "/content/drive/MyDrive/4.1_4.2_input.txt"

TF_CSV_41 = "/content/term_frequency_4_1.csv"
NER_CSV_41 = "/content/ner_results_4_1.csv"


In [4]:
import re
import csv
from collections import Counter

import pandas as pd
import spacy

with open(INPUT_FILE_41, "r", encoding="utf-8") as f:
    text_41 = f.read()

terms_41 = re.findall(r"\b[a-zA-Z0-9]+(?:['-][a-zA-Z0-9]+)*\b", text_41.lower())
freq_41 = Counter(terms_41)

tf_df_41 = pd.DataFrame(freq_41.items(), columns=["Term", "Frequency"])
tf_df_41 = tf_df_41.sort_values(
    by=["Frequency", "Term"], ascending=[False, True]
).reset_index(drop=True)

tf_df_41.to_csv(TF_CSV_41, index=False)

print(f"Total unique terms: {len(tf_df_41)}")
print(f"TF output saved to: {TF_CSV_41}")

print("\nTop 10 most frequent terms:")
display(tf_df_41.head(10))


Total unique terms: 750
TF output saved to: /content/term_frequency_4_1.csv

Top 10 most frequent terms:


,Term,Frequency
0,a,90
1,and,86
2,the,75
3,in,42
4,to,42
5,can,41
6,of,40
7,is,37
8,may,30
9,word,30


In [5]:
nlp = spacy.load("en_core_web_sm")
doc = nlp(text_41)

entities_41 = [
    {
        "Entity": ent.text,
        "Label": ent.label_,
        "Description": spacy.explain(ent.label_) or ""
    }
    for ent in doc.ents
]

ner_df_41 = pd.DataFrame(entities_41)

if ner_df_41.empty:
    print("No named entities were detected.")
else:
    display(ner_df_41)
    ner_df_41.to_csv(NER_CSV_41, index=False)
    print(f"NER output saved to: {NER_CSV_41}")


,Entity,Label,Description
0,Natural Language Processing and Artificial Int...,ORG,"Companies, agencies, institutions, etc."
1,NLP,ORG,"Companies, agencies, institutions, etc."
2,English,LANGUAGE,Any named language
3,Hindi,GPE,"Countries, cities, states"
4,Assamese,NORP,Nationalities or religious or political groups
...,...,...,...
61,NLP,ORG,"Companies, agencies, institutions, etc."
62,Dataset,ORG,"Companies, agencies, institutions, etc."
63,NLP,ORG,"Companies, agencies, institutions, etc."
64,NLP,ORG,"Companies, agencies, institutions, etc."


NER output saved to: /content/ner_results_4_1.csv


## 4.2 — Term-Frequency Analysis without using any NLP toolkit

This section uses only Python standard-library functionality for tokenization, counting, sorting, and CSV generation.


In [6]:
INPUT_FILE_42 = "/content/drive/MyDrive/4.1_4.2_input.txt"
TF_CSV_42 = "/content/term_frequency_4_2.csv"

In [7]:
import re
import csv
from collections import Counter

with open(INPUT_FILE_42, "r", encoding="utf-8") as f:
    text_42 = f.read()

words_42 = re.findall(r"\b[a-zA-Z0-9]+(?:['-][a-zA-Z0-9]+)*\b", text_42.lower())

frequency_42 = Counter(words_42)

rows_42 = sorted(frequency_42.items(), key=lambda x: (-x[1], x[0]))

with open(TF_CSV_42, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Term", "Frequency"])
    writer.writerows(rows_42)

print(f"Total unique terms: {len(rows_42)}")
print(f"TF output saved to: {TF_CSV_42}")

print("\nTop 10 most frequent terms:")
for term, count in rows_42[:10]:
    print(f"{term}: {count}")


Total unique terms: 750
TF output saved to: /content/term_frequency_4_2.csv

Top 10 most frequent terms:
a: 90
and: 86
the: 75
in: 42
to: 42
can: 41
of: 40
is: 37
may: 30
word: 30


In [8]:
tf_df_42 = pd.DataFrame(rows_42, columns=["Term", "Frequency"])
display(tf_df_42)


,Term,Frequency
0,a,90
1,and,86
2,the,75
3,in,42
4,to,42
...,...,...
745,word2vec,1
746,wording,1
747,work,1
748,worked,1


## 4.3 — TF, DF, IDF and TF-IDF without an NLP toolkit

**Tasks covered:** read multiple documents, lowercase, remove punctuation, tokenize, calculate TF/DF/IDF/TF-IDF, display the values, and identify the top 10 TF-IDF terms for each document.

**IDF formula used:**

$$
IDF(t) = \log\left(\frac{N}{DF(t)}\right)
$$

where `N` is the total number of documents.

**TF formula used:**

$$
TF(t,d) = \frac{\text{count of term }t\text{ in document }d}{\text{total terms in document }d}
$$

**TF-IDF:**

$$
TFIDF(t,d) = TF(t,d) \times IDF(t)
$$


In [9]:
DATASET_DIR_43 = "/content/drive/MyDrive/Experiment_4_3_Text_Documents"

OUTPUT_DIR_43 = "/content/tf_idf_outputs"

ALL_RESULTS_CSV_43 = f"{OUTPUT_DIR_43}/tf_df_idf_tfidf_all.csv"
TOP10_CSV_43 = f"{OUTPUT_DIR_43}/top10_tfidf_terms.csv"

In [11]:
import os
import re
import math
import pandas as pd
from collections import Counter

os.makedirs(OUTPUT_DIR_43, exist_ok=True)

doc_files_43 = sorted(
    os.path.join(DATASET_DIR_43, name)
    for name in os.listdir(DATASET_DIR_43)
    if name.lower().endswith(".txt")
)

if not doc_files_43:
    raise FileNotFoundError(
        f"No .txt files found in {DATASET_DIR_43}. "
        "Upload your documents and update DATASET_DIR_43."
    )

print(f"Found {len(doc_files_43)} document(s):")
for path in doc_files_43:
    print(" -", os.path.basename(path))


Found 3 document(s):
 - Document_1.txt
 - Document_2.txt
 - Document_3.txt


In [13]:
documents_43 = {}

for path in doc_files_43:
    name = os.path.splitext(os.path.basename(path))[0]

    with open(path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    text = raw_text.lower()

    text = re.sub(r"[^a-z0-9\s]", " ", text)

    tokens = re.findall(r"\b[a-z0-9]+\b", text)

    documents_43[name] = tokens

print("\nPreprocessed documents:")
for name, tokens in documents_43.items():
    print(f"{name}: {len(tokens)} tokens")



Preprocessed documents:
Document_1: 9 tokens
Document_2: 8 tokens
Document_3: 9 tokens


In [14]:
tf_by_doc = {}

for doc_name, tokens in documents_43.items():
    counts = Counter(tokens)
    total_terms = len(tokens)

    tf_by_doc[doc_name] = {
        term: count / total_terms
        for term, count in counts.items()
    }

tf_rows = []
for doc_name, tf_values in tf_by_doc.items():
    for term, tf in tf_values.items():
        tf_rows.append({
            "Document": doc_name,
            "Term": term,
            "TF": tf
        })

tf_df = pd.DataFrame(tf_rows).sort_values(
    by=["Document", "TF", "Term"],
    ascending=[True, False, True]
).reset_index(drop=True)

print("TF values:")
display(tf_df)


TF values:


,Document,Term,TF
0,Document_1,a,0.111111
1,Document_1,artificial,0.111111
2,Document_1,field,0.111111
3,Document_1,intelligence,0.111111
4,Document_1,is,0.111111
5,Document_1,language,0.111111
6,Document_1,natural,0.111111
7,Document_1,of,0.111111
8,Document_1,processing,0.111111
9,Document_2,language,0.250000


In [15]:
num_documents = len(documents_43)

df_counts = Counter()

for tokens in documents_43.values():
    for term in set(tokens):
        df_counts[term] += 1

df_df = pd.DataFrame(
    [{"Term": term, "DF": df} for term, df in df_counts.items()]
).sort_values(
    by=["DF", "Term"], ascending=[False, True]
).reset_index(drop=True)

print("DF values:")
display(df_df)


DF values:


,Term,DF
0,artificial,2
1,intelligence,2
2,is,2
3,language,2
4,natural,2
5,of,2
6,processing,2
7,a,1
8,an,1
9,computers,1


In [16]:
idf_df = df_df.copy()
idf_df["IDF"] = idf_df["DF"].apply(
    lambda df: math.log(num_documents / df)
)

print("IDF values:")
display(idf_df[["Term", "DF", "IDF"]])


IDF values:


,Term,DF,IDF
0,artificial,2,0.405465
1,intelligence,2,0.405465
2,is,2,0.405465
3,language,2,0.405465
4,natural,2,0.405465
5,of,2,0.405465
6,processing,2,0.405465
7,a,1,1.098612
8,an,1,1.098612
9,computers,1,1.098612


In [17]:
idf_lookup = dict(zip(idf_df["Term"], idf_df["IDF"]))

tfidf_rows = []

for doc_name, tf_values in tf_by_doc.items():
    for term, tf in tf_values.items():
        idf = idf_lookup[term]
        tfidf = tf * idf

        tfidf_rows.append({
            "Document": doc_name,
            "Term": term,
            "TF": tf,
            "DF": df_counts[term],
            "IDF": idf,
            "TF-IDF": tfidf
        })

tfidf_df = pd.DataFrame(tfidf_rows).sort_values(
    by=["Document", "TF-IDF", "Term"],
    ascending=[True, False, True]
).reset_index(drop=True)

print("TF, DF, IDF and TF-IDF:")
display(tfidf_df)


TF, DF, IDF and TF-IDF:


,Document,Term,TF,DF,IDF,TF-IDF
0,Document_1,a,0.111111,1,1.098612,0.122068
1,Document_1,field,0.111111,1,1.098612,0.122068
2,Document_1,artificial,0.111111,2,0.405465,0.045052
3,Document_1,intelligence,0.111111,2,0.405465,0.045052
4,Document_1,is,0.111111,2,0.405465,0.045052
5,Document_1,language,0.111111,2,0.405465,0.045052
6,Document_1,natural,0.111111,2,0.405465,0.045052
7,Document_1,of,0.111111,2,0.405465,0.045052
8,Document_1,processing,0.111111,2,0.405465,0.045052
9,Document_2,computers,0.125000,1,1.098612,0.137327


In [18]:
tfidf_df.to_csv(ALL_RESULTS_CSV_43, index=False)

print(f"Complete TF-IDF table saved to: {ALL_RESULTS_CSV_43}")


Complete TF-IDF table saved to: /content/tf_idf_outputs/tf_df_idf_tfidf_all.csv


In [19]:
top10_rows = []

for doc_name in documents_43:
    top10 = (
        tfidf_df[tfidf_df["Document"] == doc_name]
        .sort_values(by=["TF-IDF", "Term"], ascending=[False, True])
        .head(10)
        .copy()
    )

    print(f"\nTop 10 TF-IDF terms — {doc_name}")
    display(top10[["Term", "TF", "DF", "IDF", "TF-IDF"]])

    for rank, (_, row) in enumerate(top10.iterrows(), start=1):
        top10_rows.append({
            "Document": doc_name,
            "Rank": rank,
            "Term": row["Term"],
            "TF": row["TF"],
            "DF": row["DF"],
            "IDF": row["IDF"],
            "TF-IDF": row["TF-IDF"]
        })

top10_df = pd.DataFrame(top10_rows)
top10_df.to_csv(TOP10_CSV_43, index=False)

print(f"Top-10 results saved to: {TOP10_CSV_43}")



Top 10 TF-IDF terms — Document_1


,Term,TF,DF,IDF,TF-IDF
0,a,0.111111,1,1.098612,0.122068
1,field,0.111111,1,1.098612,0.122068
2,artificial,0.111111,2,0.405465,0.045052
3,intelligence,0.111111,2,0.405465,0.045052
4,is,0.111111,2,0.405465,0.045052
5,language,0.111111,2,0.405465,0.045052
6,natural,0.111111,2,0.405465,0.045052
7,of,0.111111,2,0.405465,0.045052
8,processing,0.111111,2,0.405465,0.045052



Top 10 TF-IDF terms — Document_2


,Term,TF,DF,IDF,TF-IDF
9,computers,0.125,1,1.098612,0.137327
10,helps,0.125,1,1.098612,0.137327
11,human,0.125,1,1.098612,0.137327
12,understand,0.125,1,1.098612,0.137327
13,language,0.250,2,0.405465,0.101366
14,natural,0.125,2,0.405465,0.050683
15,processing,0.125,2,0.405465,0.050683



Top 10 TF-IDF terms — Document_3


,Term,TF,DF,IDF,TF-IDF
16,an,0.111111,1,1.098612,0.122068
17,important,0.111111,1,1.098612,0.122068
18,learning,0.111111,1,1.098612,0.122068
19,machine,0.111111,1,1.098612,0.122068
20,part,0.111111,1,1.098612,0.122068
21,artificial,0.111111,2,0.405465,0.045052
22,intelligence,0.111111,2,0.405465,0.045052
23,is,0.111111,2,0.405465,0.045052
24,of,0.111111,2,0.405465,0.045052


Top-10 results saved to: /content/tf_idf_outputs/top10_tfidf_terms.csv


## Output files

After running the notebook, the following CSV files will be available in the Colab environment:

- `term_frequency_4_1.csv`
- `ner_results_4_1.csv`
- `term_frequency_4_2.csv`
- `tf_df_idf_tfidf_all.csv`
- `top10_tfidf_terms.csv`

For **4.1 and 4.2**, use the same uploaded input text file or change the path independently.

For **4.3**, place all the text documents inside the folder given by `DATASET_DIR_43`.
